# ~ TilStat scraper ~

Utility to scrape the scores to enter the various courses over the years. Like any scrape utility, this code sucks.

In [1]:
import requests
from bs4 import BeautifulSoup as bs
from multiprocessing import Pool
import pandas as pd
import numpy as np

In [2]:
from datetime import datetime

## 2017/18 - 2020/21

In [3]:
base = 'http://didattica.polito.it'

In [4]:
indexes = []

for year in [2018,2019,2020,2021]:
    indexes += [
        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.arc?p_a_acc={year}',
         'conting': False, 'course': 'Architettura', 'year':year},
        # {'uri':f'{base}/pls/portal30/preimma.pkg_grad.arc_conting?p_a_acc={year}',
        #  'conting': True, 'course': 'Architettura', 'year':year},

        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.ing?p_a_acc={year}',
         'conting': False, 'course': 'Ingegneria', 'year':year},
        # {'uri':f'{base}/pls/portal30/preimma.pkg_grad.ing_cont?p_a_acc={year}',
        #  'conting': True, 'course': 'Ingegneria', 'year':year},

        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.des?p_a_acc={year}',
         'conting': False, 'course': 'Design', 'year':year},
        # {'uri':f'{base}/pls/portal30/preimma.pkg_grad.des_cont?p_a_acc={year}',
        #  'conting': True, 'course': 'Design', 'year':year},

        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.pian?p_a_acc={year}',
         'conting': False, 'course': 'Pianificazione', 'year':year},
        # {'uri':f'{base}/pls/portal30/preimma.pkg_grad.pian_cont?p_a_acc={year}',
        #  'conting': True, 'course': 'Pianificazione', 'year':year}, 
    ]

In [5]:
indexes

[{'uri': 'http://didattica.polito.it/pls/portal30/preimma.pkg_grad.arc?p_a_acc=2018',
  'conting': False,
  'course': 'Architettura',
  'year': 2018},
 {'uri': 'http://didattica.polito.it/pls/portal30/preimma.pkg_grad.ing?p_a_acc=2018',
  'conting': False,
  'course': 'Ingegneria',
  'year': 2018},
 {'uri': 'http://didattica.polito.it/pls/portal30/preimma.pkg_grad.des?p_a_acc=2018',
  'conting': False,
  'course': 'Design',
  'year': 2018},
 {'uri': 'http://didattica.polito.it/pls/portal30/preimma.pkg_grad.pian?p_a_acc=2018',
  'conting': False,
  'course': 'Pianificazione',
  'year': 2018},
 {'uri': 'http://didattica.polito.it/pls/portal30/preimma.pkg_grad.arc?p_a_acc=2019',
  'conting': False,
  'course': 'Architettura',
  'year': 2019},
 {'uri': 'http://didattica.polito.it/pls/portal30/preimma.pkg_grad.ing?p_a_acc=2019',
  'conting': False,
  'course': 'Ingegneria',
  'year': 2019},
 {'uri': 'http://didattica.polito.it/pls/portal30/preimma.pkg_grad.des?p_a_acc=2019',
  'conting': Fa

In [6]:
# indexes = []

for year in [2022, 2023, 2024]:
    indexes += [
        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.arc?p_a_acc={year}',
         'conting': False, 'course': 'Architettura', 'year':year},
        # {'uri':f'{base}/pls/portal30/preimma.pkg_grad.arc_conting?p_a_acc={year}',
        #  'conting': True, 'course': 'Architettura', 'year':year},

        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.des?p_a_acc={year}',
         'conting': False, 'course': 'Design', 'year':year},
        # {'uri':f'{base}/pls/portal30/preimma.pkg_grad.des_cont?p_a_acc={year}',
        #  'conting': True, 'course': 'Design', 'year':year},

        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.pian?p_a_acc={year}',
         'conting': False, 'course': 'Pianificazione', 'year':year},
        # {'uri':f'{base}/pls/portal30/preimma.pkg_grad.pian_cont?p_a_acc={year}',
        #  'conting': True, 'course': 'Pianificazione', 'year':year}, 
    ]

In [7]:
ranks = []
for index in indexes:
    html = requests.get(index['uri']).text
    soup = bs(html, 'html.parser')
    ranklinks = [link for link in soup.find_all("a")
                     if "href" in link.attrs and 'preimma' in link['href'] and link.text.startswith('Graduatoria del ')]
    
    for i, link in enumerate(ranklinks):
        
        # paranoid check
        # Es: Ingegneria has .ing string in each rank
        assert f".{index['course'].lower()[:3]}" in link['href'], "Wrong link assoc..."
        
        date = [int(x) for x in link.text.replace("Graduatoria del ","").split('/')]
        date = datetime(date[2],date[1],date[0])
        
        # paranoid check
        first_sane_rank = datetime(int(index['year'])-1, 7, 1) # dal primo luglio del primo anno di inizio coorte
        last_sane_rank = datetime(int(index['year']), 3, 15) # a metà marzo del secondo anno di inizio coorte
        assert first_sane_rank < date < last_sane_rank, "Wrong year assoc..."
        
        ranks.append({
            'uri': base+link['href'],
            'conting': index['conting'],
            'course': index['course'],
            'grad_n': i,
            'date': date,
            'year': index['year']
        })

In [8]:
x=0
def scrapeRank(rank):
    global x
    x+=1
    print(x)
    
    html = requests.get(rank['uri']).text
    soup = bs(html, 'html.parser')
    
    headers = soup.select("table th")
    rows = soup.select("table tr")
    cells = soup.select("table td")

    # paranoid check
    assert len(headers)*(len(rows)-1) == len(cells), "Wrong table format"

    headervalues = [str(h.text.lower().strip()) for h in headers]
    values = [str(c.text.strip()) for c in cells]
    shapedvalues = np.asarray(values).reshape((len(rows)-1),len(headers)).astype(str)
    df = pd.DataFrame(shapedvalues, columns=headervalues)

    df = df[df['totale'] != "Certificazione Internazionale"]

    df["totale"] = df["totale"].str.replace(',','.').astype(float)
    df['uri'] = rank['uri']
    df['conting'] = rank['conting']
    df['course'] = rank['course']
    df['grad_n'] = rank['grad_n']
    df['date'] = rank['date']
    df['year'] = rank['year']
    

    
    return df

In [9]:
len(ranks)

160

In [10]:
dfs = list(map(scrapeRank, ranks))

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160


In [11]:
result = pd.concat(dfs, ignore_index=True)

In [12]:
result.to_csv(f"ranks/pre2022_{str(datetime.now()).replace(':','-')}.csv")